# Simulación Efecto Diodo Superconductor — Peine Asimétrico
### Paralelizado | 130 trabajos | 64 núcleos
---
**Variables:**
- N peines: 2, 4, 6, 8, 10
- Campo B: 0.0 a 1.2 mT (pasos de 0.1)
- Polaridades: positivo y negativo (invertir source/drain)
- Barrido corriente: 0 a 50 µA (pasos de 0.1)

In [15]:
# ═══════════════════════════════════════════════════
# CELDA 1 — IMPORTS
# ═══════════════════════════════════════════════════
%config InlineBackend.figure_formats = {"retina", "png"}
%matplotlib inline

import os
import time
import tempfile
import shutil
import itertools
import threading
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Backend sin pantalla, necesario para paralelización
import matplotlib.pyplot as plt

from joblib import Parallel, delayed

import tdgl
from tdgl.geometry import box

print("Imports completados correctamente.")

Imports completados correctamente.


In [16]:
# ═══════════════════════════════════════════════════
# CELDA 2 — PARÁMETROS GLOBALES

# ═══════════════════════════════════════════════════
TEMP_DIR = "/media/david/ea1787b7-95f4-4875-ace4-8e79eb830ea3/home/david/Documents/Problema 3"
# ─────────────────────────────────────────────
# PARÁMETROS DEL MATERIAL
# ─────────────────────────────────────────────
XI            = 0.5   # Longitud de coherencia (µm)
LONDON_LAMBDA = 2.0   # Longitud de penetración de London (µm)
D_THICKNESS   = 0.1   # Espesor de la lámina (µm)
GAMMA         = 1.0   # Parámetro de amortiguamiento

# ─────────────────────────────────────────────
# PARÁMETROS GEOMÉTRICOS FIJOS
# Según el paper: Lx=20 µm, Ly=5 µm, hd=3.5 µm
# factor_escala = 1.0 (sin escalar)
# ─────────────────────────────────────────────
TOTAL_LENGTH = 20.0   # Lx: longitud total del dispositivo (µm)
TOTAL_WIDTH  =  5.0   # Ly: altura del bloque rectangular base (µm)
HD           =  3.5   # Profundidad máxima de los dientes (µm)

# ─────────────────────────────────────────────
# CONTACTOS: totalidad de la pared lateral
# ─────────────────────────────────────────────
CONTACT_LENGTH = TOTAL_WIDTH   # Toda la pared lateral (µm)
CONTACT_WIDTH  = 0.1           # Ancho del contacto en x (µm)

# ─────────────────────────────────────────────
# PARÁMETROS DE SIMULACIÓN
# ─────────────────────────────────────────────
N_PEINES    = [2, 4, 6, 8, 10]                              # Número de crestas del peine
B_VALORES   = [round(b, 1) for b in np.arange(0.0, 1.3, 0.1)]  # Campo magnético externo (mT)
POLARIDADES = ["positivo", "negativo"]                      # Polaridad de corriente

# ─────────────────────────────────────────────
# BARRIDO DE CORRIENTE
# ─────────────────────────────────────────────
CORRIENTE_MIN  =  0.0   # Corriente mínima del barrido (µA)
CORRIENTE_MAX  = 50.0   # Corriente máxima del barrido (µA)
CORRIENTE_PASO =  0.1   # Paso del barrido (µA)

# ─────────────────────────────────────────────
# OPCIONES DEL SOLVER
# ─────────────────────────────────────────────
SKIP_TIME  = 100   # Tiempo de estabilización antes de medir (u.a.)
SOLVE_TIME = 200   # Tiempo de simulación activa (u.a.)

# ─────────────────────────────────────────────
# PARALELIZACIÓN
# ─────────────────────────────────────────────
N_JOBS = 60   # Núcleos a usar (de 128 disponibles)

# ─────────────────────────────────────────────
# CARPETA RAÍZ DE RESULTADOS
# ─────────────────────────────────────────────
RESULTS_DIR = "Results"
CSV_PATH    = os.path.join(RESULTS_DIR, "resultados_globales.csv")
TXT_PATH    = os.path.join(RESULTS_DIR, "parametros_simulacion.txt")

# Lista completa de trabajos
TRABAJOS = list(itertools.product(N_PEINES, B_VALORES, POLARIDADES))

print(f"Total de trabajos a correr: {len(TRABAJOS)}")
print(f"N_peines: {N_PEINES}")
print(f"B_valores: {B_VALORES}")
print(f"Polaridades: {POLARIDADES}")
print(f"Núcleos a usar: {N_JOBS}")

Total de trabajos a correr: 130
N_peines: [2, 4, 6, 8, 10]
B_valores: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
Polaridades: ['positivo', 'negativo']
Núcleos a usar: 60


In [17]:
# ═══════════════════════════════════════════════════
# CELDA 3 — CREAR ESTRUCTURA DE CARPETAS Y TXT
# ═══════════════════════════════════════════════════

def crear_estructura_carpetas():
    """Crea toda la estructura Results/Nx_crestas/polaridad/B-x.x/"""
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for N in N_PEINES:
        for pol in POLARIDADES:
            for B in B_VALORES:
                ruta = os.path.join(
                    RESULTS_DIR,
                    f"N{N}_crestas",
                    pol,
                    f"B-{B:.1f}"
                )
                os.makedirs(ruta, exist_ok=True)
    print(f"Estructura de carpetas creada en: {RESULTS_DIR}/")

def guardar_txt_parametros():
    """Guarda un TXT con todos los parámetros de simulación."""
    with open(TXT_PATH, "w", encoding="utf-8") as f:
        f.write("=" * 55 + "\n")
        f.write("  PARÁMETROS DE SIMULACIÓN\n")
        f.write(f"  Fecha inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 55 + "\n\n")

        f.write("--- MATERIAL ---\n")
        f.write(f"xi (coherencia)          = {XI} µm\n")
        f.write(f"london_lambda            = {LONDON_LAMBDA} µm\n")
        f.write(f"d (espesor)              = {D_THICKNESS} µm\n")
        f.write(f"gamma                    = {GAMMA}\n\n")

        f.write("--- GEOMETRÍA FIJA ---\n")
        f.write(f"Lx (longitud total)      = {TOTAL_LENGTH} µm\n")
        f.write(f"Ly (altura bloque)       = {TOTAL_WIDTH} µm\n")
        f.write(f"hd (prof. dientes)       = {HD} µm\n")
        f.write(f"Pared restante (Ly-hd)   = {TOTAL_WIDTH - HD} µm\n")
        f.write(f"contact_length           = {CONTACT_LENGTH} µm (toda la pared)\n")
        f.write(f"contact_width            = {CONTACT_WIDTH} µm\n\n")

        f.write("--- VARIABLES DE SIMULACIÓN ---\n")
        f.write(f"N peines                 = {N_PEINES}\n")
        f.write(f"Campo B (mT)             = {B_VALORES}\n")
        f.write(f"Polaridades              = {POLARIDADES}\n")
        f.write(f"Total trabajos           = {len(TRABAJOS)}\n\n")

        f.write("--- BARRIDO DE CORRIENTE ---\n")
        f.write(f"Corriente min            = {CORRIENTE_MIN} µA\n")
        f.write(f"Corriente max            = {CORRIENTE_MAX} µA\n")
        f.write(f"Paso                     = {CORRIENTE_PASO} µA\n")
        f.write(f"Número de puntos         = {len(np.arange(CORRIENTE_MIN, CORRIENTE_MAX, CORRIENTE_PASO))}\n\n")

        f.write("--- SOLVER ---\n")
        f.write(f"skip_time                = {SKIP_TIME}\n")
        f.write(f"solve_time               = {SOLVE_TIME}\n")
        f.write(f"max_edge_length          = xi/2 = {XI/2} µm\n")
        f.write(f"smooth                   = 100\n\n")

        f.write("--- PARALELIZACIÓN ---\n")
        f.write(f"CPUs disponibles         = 128\n")
        f.write(f"n_jobs                   = {N_JOBS}\n")

    print(f"Parámetros guardados en: {TXT_PATH}")

crear_estructura_carpetas()
guardar_txt_parametros()

Estructura de carpetas creada en: Results/
Parámetros guardados en: Results/parametros_simulacion.txt


In [18]:
# ═══════════════════════════════════════════════════
# CELDA 4 — FUNCIÓN: CONSTRUIR DISPOSITIVO
# ═══════════════════════════════════════════════════

def construir_dispositivo(N):
    """
    Construye el dispositivo superconductor de peine asimétrico con N dientes.
    El contacto ocupa la totalidad de la pared lateral.
    Retorna el device con malla generada.
    """
    # Parámetros de los dientes gaussianos (Ec. 1 del paper)
    delta_x = TOTAL_LENGTH / N          # Separación entre centros de dientes (µm)
    delta   = 0.20 * delta_x            # Parámetro de suavidad (δ = 0.20·Δx)
    x_teeth = np.linspace(
        -TOTAL_LENGTH / 2 + delta_x / 2,
         TOTAL_LENGTH / 2 - delta_x / 2,
        N
    )

    # Perfil del borde inferior con dientes gaussianos
    def lower_edge_profile(x):
        y = np.zeros_like(x, dtype=float)
        for xi_i in x_teeth:
            y += -HD * np.exp(-((x - xi_i) ** 2) / (2 * delta ** 2))
        return y

    # Construcción del polígono
    n_pts    = 1000
    x_bottom = np.linspace(-TOTAL_LENGTH / 2, TOTAL_LENGTH / 2, n_pts)
    y_bottom = lower_edge_profile(x_bottom)
    y_top    =  TOTAL_WIDTH / 2
    y_base   = -TOTAL_WIDTH / 2
    y_bottom_coords = y_base + y_bottom

    x_polygon = np.concatenate([
        np.array([-TOTAL_LENGTH/2, TOTAL_LENGTH/2]),   # borde superior
        np.array([TOTAL_LENGTH/2, TOTAL_LENGTH/2]),    # lado derecho
        x_bottom[::-1],                                 # borde inferior (dientes)
        np.array([-TOTAL_LENGTH/2, -TOTAL_LENGTH/2])   # lado izquierdo
    ])
    y_polygon = np.concatenate([
        np.array([y_top, y_top]),
        np.array([y_top, y_base]),
        y_bottom_coords[::-1],
        np.array([y_base, y_top])
    ])

    polygon_points = np.column_stack([x_polygon, y_polygon])

    # Film
    film = (
        tdgl.Polygon("film", points=polygon_points)
        .resample(1200)
        .buffer(0)
    )

    # Terminales: contacto ocupa toda la pared lateral
    source = (
        tdgl.Polygon("source", points=box(CONTACT_WIDTH, CONTACT_LENGTH))
        .translate(dx=-TOTAL_LENGTH / 2)
    )
    drain = source.scale(xfact=-1).set_name("drain")

    # Puntos de medición de voltaje
    probe_points = [
        (-TOTAL_LENGTH / 2.5, 0),
         (TOTAL_LENGTH / 2.5, 0)
    ]

    # Layer
    layer = tdgl.Layer(
        coherence_length=XI,
        london_lambda=LONDON_LAMBDA,
        thickness=D_THICKNESS,
        gamma=GAMMA
    )

    # Device
    device = tdgl.Device(
        f"peine_N{N}",
        layer=layer,
        film=film,
        terminals=[source, drain],
        probe_points=probe_points,
        length_units="um",
    )

    # Generar malla
    device.make_mesh(max_edge_length=XI / 2, smooth=100)

    return device

print("Función construir_dispositivo() definida.")

Función construir_dispositivo() definida.


In [19]:
# ═══════════════════════════════════════════════════
# CELDA 5 — FUNCIONES AUXILIARES
# ═══════════════════════════════════════════════════

def get_terminal_currents(current, polaridad):
    if polaridad == "positivo":
        return {"source":  current, "drain": -current}
    else:
        return {"source": -current, "drain":  current}


def detectar_corrientes_criticas(voltages, currents):
    gaps = [(voltages[i+1] - voltages[i], i) for i in range(len(voltages) - 1)]
    gaps_sorted = sorted(gaps, key=lambda x: x[0], reverse=True)

    # Curva plana: no hay corriente crítica
    rango = max(voltages) - min(voltages)
    if rango < 0.01:
        return None, None

    umbral = rango * 0.05
    saltos_significativos = [(g, i) for g, i in gaps_sorted if g > umbral]

    if len(saltos_significativos) == 0:
        return None, None
    elif len(saltos_significativos) == 1:
        idx1 = saltos_significativos[0][1]
        return currents[idx1], None
    else:
        indices = sorted([s[1] for s in saltos_significativos[:2]])
        return currents[indices[0]], currents[indices[1]]


def guardar_fila_csv(fila_dict):
    """Cada proceso escribe su propio archivo parcial — sin lock."""
    pid = os.getpid()
    ruta_parcial = os.path.join(RESULTS_DIR, f"parcial_{pid}.csv")
    fila_df = pd.DataFrame([fila_dict])
    header  = not os.path.exists(ruta_parcial)
    fila_df.to_csv(ruta_parcial, mode='a', header=header, index=False)

print("Funciones auxiliares definidas.")

Funciones auxiliares definidas.


In [20]:
# ═══════════════════════════════════════════════════
# CELDA 6 — FUNCIÓN PRINCIPAL: CORRER SIMULACIÓN
# ═══════════════════════════════════════════════════

def correr_simulacion(N, B, polaridad):
    """
    Corre una simulación completa para (N, B, polaridad):
    1. Construye el dispositivo
    2. Simulación espacial → mapas de corrientes y |ψ|²
    3. Barrido V-I → curva + Jc1 + Jc2
    4. Guarda todas las gráficas y actualiza el CSV global
    """
    os.environ["OPENBLAS_NUM_THREADS"] = "2"
    label = f"[N={N}, B={B:.1f}mT, {polaridad}]"
    t_inicio_total = time.time()

    # Carpeta de salida para este trabajo
    ruta_salida = os.path.join(
        RESULTS_DIR, f"N{N}_crestas", polaridad, f"B-{B:.1f}"
    )

    # Carpeta temporal propia para el .h5 (evita colisiones entre procesos)
    tmpdir = tempfile.mkdtemp(dir=TEMP_DIR)

    try:
        # ── Construir dispositivo ──────────────────────
        device = construir_dispositivo(N)

        # ── Opciones del solver ────────────────────────
        h5_path = os.path.join(tmpdir, f"sim_N{N}_B{B}_{polaridad}.h5")
        options = tdgl.SolverOptions(
            skip_time=SKIP_TIME,
            solve_time=SOLVE_TIME,
            output_file=h5_path,
            field_units="mT",
            current_units="uA",
            save_every=100,
        )

        # ── Campo magnético externo perpendicular ──────
        # H aplicado en dirección ẑ (perpendicular al film)
        campo_B = tdgl.sources.ConstantField(B, field_units="mT")

        # ══════════════════════════════════════════════
        # 1. SIMULACIÓN ESPACIAL (corriente fija = 12 µA)
        # ══════════════════════════════════════════════
        t0 = time.time()

        solucion_espacial = tdgl.solve(
            device=device,
            options=options,
            applied_vector_potential=campo_B,
            terminal_currents=get_terminal_currents(12.0, polaridad)
        )

        t_espacial = time.time() - t0
        print(f"{label} Simulación espacial: {t_espacial:.1f}s")

        # Mapa de corrientes
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
        _ = solucion_espacial.plot_currents(ax=axes[0], streamplot=False)
        _ = solucion_espacial.plot_currents(ax=axes[1])
        fig.suptitle(f'Corrientes — N={N}, B={B:.1f}mT, {polaridad}')
        fig.savefig(os.path.join(ruta_salida, "mapa_corrientes.png"), dpi=150)
        plt.close(fig)

        # Densidad de pares de Cooper |ψ|²
        fig, axes_op = solucion_espacial.plot_order_parameter()
        axes_op.flat[0].set_title(f'|ψ|² — N={N}, B={B:.1f}mT, {polaridad}')
        fig.savefig(os.path.join(ruta_salida, "densidad_cooper.png"), dpi=150)
        plt.close(fig)

        # ══════════════════════════════════════════════
        # 2. BARRIDO V-I
        # ══════════════════════════════════════════════
        t0 = time.time()

        corrientes  = np.arange(CORRIENTE_MIN, CORRIENTE_MAX, CORRIENTE_PASO)
        voltajes    = []
        solucion_anterior = None

        for current in corrientes:
            # Nuevo h5 por punto del barrido
            h5_barrido = os.path.join(tmpdir, "barrido_actual.h5")
            if os.path.exists(h5_barrido):
                os.remove(h5_barrido)
                
            opts_barrido = tdgl.SolverOptions(
                skip_time=SKIP_TIME,
                solve_time=SOLVE_TIME,
                output_file=h5_barrido,
                field_units="mT",
                current_units="uA",
                save_every=100,
            )
            sol = tdgl.solve(
                device=device,
                options=opts_barrido,
                applied_vector_potential=campo_B,
                terminal_currents=get_terminal_currents(current, polaridad),
                seed_solution=solucion_anterior   # estado anterior como inicial
            )
            voltajes.append(sol.dynamics.mean_voltage())
            solucion_anterior = sol

        t_barrido = time.time() - t0
        print(f"{label} Barrido V-I: {t_barrido:.1f}s")

        # ── Detectar corrientes críticas ───────────────
        Jc1, Jc2 = detectar_corrientes_criticas(voltajes, corrientes)

        # ── Gráfica curva V-I ──────────────────────────
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(corrientes, voltajes, color='blue', linewidth=1)
        if Jc1 is not None:
            ax.axvline(Jc1, color='red',    linestyle='--', label=f'Jc1 = {Jc1:.2f} µA')
        if Jc2 is not None:
            ax.axvline(Jc2, color='orange', linestyle='--', label=f'Jc2 = {Jc2:.2f} µA')
        ax.set_xlabel('Corriente (µA)')
        ax.set_ylabel('Voltaje promedio (V)')
        ax.set_title(f'Curva V-I — N={N}, B={B:.1f}mT, {polaridad}')
        ax.grid(True)
        ax.legend()
        plt.tight_layout()
        fig.savefig(os.path.join(ruta_salida, "curva_VI.png"), dpi=150)
        plt.close(fig)

        # ── Guardar CSV del barrido local ──────────────
        csv_local = pd.DataFrame({'current_uA': corrientes, 'voltage_V': voltajes})
        csv_local.to_csv(os.path.join(ruta_salida, "VI_data.csv"), index=False)

        # ══════════════════════════════════════════════
        # 3. GUARDAR FILA EN CSV GLOBAL
        # ══════════════════════════════════════════════
        t_total = time.time() - t_inicio_total

        fila = {
            "N_peine":    N,
            "polaridad":  polaridad,
            "B_mT":       B,
            "Jc1_uA":     Jc1 if Jc1 is not None else np.nan,
            "Jc2_uA":     Jc2 if Jc2 is not None else np.nan,
            "eta_c1_pct": np.nan,   # se calcula en post-procesamiento
            "eta_c2_pct": np.nan,   # se calcula en post-procesamiento
            "tiempo_s":   round(t_total, 1),
            "estado":     "OK"
        }
        guardar_fila_csv(fila)

        print(f"{label} TOTAL: {t_total:.1f}s | Jc1={Jc1} µA | Jc2={Jc2} µA")
        return fila

    except Exception as e:
        t_total = time.time() - t_inicio_total
        print(f"{label} ERROR después de {t_total:.1f}s: {e}")
        traceback.print_exc()

        fila_error = {
            "N_peine":    N,
            "polaridad":  polaridad,
            "B_mT":       B,
            "Jc1_uA":     np.nan,
            "Jc2_uA":     np.nan,
            "eta_c1_pct": np.nan,
            "eta_c2_pct": np.nan,
            "tiempo_s":   round(t_total, 1),
            "estado":     f"ERROR: {str(e)[:80]}"
        }
        guardar_fila_csv(fila_error)   # solo una vez aquí
        return fila_error

    finally:
        # Limpiar archivos temporales siempre
        shutil.rmtree(tmpdir, ignore_errors=True)

print("Función correr_simulacion() definida.")

Función correr_simulacion() definida.


In [21]:
# ═══════════════════════════════════════════════════
# CELDA 7 — LANZAR PARALELIZACIÓN
# ═══════════════════════════════════════════════════

print("=" * 55)
print(f"  INICIANDO SIMULACIÓN PARALELA")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Total trabajos: {len(TRABAJOS)}")
print(f"  Núcleos: {N_JOBS}")
print("=" * 55)

t_global_inicio = time.time()

resultados = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=5           # Muestra progreso de joblib
)(
    delayed(correr_simulacion)(N, B, polaridad)
    for N, B, polaridad in TRABAJOS
)

t_global_total = time.time() - t_global_inicio

print("\n" + "=" * 55)
print("  SIMULACIÓN PARALELA COMPLETADA")
print("=" * 55)

  INICIANDO SIMULACIÓN PARALELA
  2026-05-19 23:13:07
  Total trabajos: 130
  Núcleos: 60


[Parallel(n_jobs=60)]: Using backend LokyBackend with 60 concurrent workers.
Constructing Voronoi polygons:   8%|▊         | 810/9814 [00:00<00:03, 2745.50it/s]s]Malformed Voronoi cell surrounding boundary site 952. Try changing the number of boundary mesh sites using Polygon.resample() or Polygon.buffer(eps) where eps is 0 or a small positive float.
Malformed Voronoi cell surrounding boundary site 953. Try changing the number of boundary mesh sites using Polygon.resample() or Polygon.buffer(eps) where eps is 0 or a small positive float.
Thermalizing:   0%|          | 0/100 [00:02<3:23:19, 122.03s/tau ]Malformed Voronoi cell surrounding boundary site 952. Try changing the number of boundary mesh sites using Polygon.resample() or Polygon.buffer(eps) where eps is 0 or a small positive float.
Malformed Voronoi cell surrounding boundary site 953. Try changing the number of boundary mesh sites using Polygon.resample() or Polygon.buffer(eps) where eps is 0 or a small positive float.
Construc

[N=4, B=1.2mT, negativo] Simulación espacial: 1714.2s


Simulating:  69%|██████▊   | 137/200 [14:34<07:40,  7.31s/tau ] ]

[N=6, B=0.1mT, negativo] Simulación espacial: 1777.3s


...

[N=2, B=0.3mT, positivo] Simulación espacial: 4685.4s


Simulating:  39%|███▉      | 79/200 [07:19<03:43,  1.84s/tau ]]]]

[N=2, B=0.3mT, negativo] Simulación espacial: 4793.1s


Simulating:  72%|███████▏  | 145/200 [28:10<09:29, 10.29s/tau ]Fatal Python error: Segmentation fault

Thread 0x000072d889058640 (most recent call first):
  File "/home/david/miniconda3/lib/python3.13/threading.py", line 363 in wait
  File "/home/david/miniconda3/lib/python3.13/threading.py", line 660 in wait
  File "/media/david/ea1787b7-95f4-4875-ace4-8e79eb830ea3/home/david/Documents/Problema 3/Entornos/P3/lib/python3.13/site-packages/tqdm/_monitor.py", line 60 in run
  File "/home/david/miniconda3/lib/python3.13/threading.py", line 1044 in _bootstrap_inner
  File "/home/david/miniconda3/lib/python3.13/threading.py", line 1015 in _bootstrap

Current thread 0x000072d8b0c21440 (most recent call first):
  File "/media/david/ea1787b7-95f4-4875-ace4-8e79eb830ea3/home/david/Documents/Problema 3/Entornos/P3/lib/python3.13/site-packages/h5py/_hl/dataset.py", line 177 in make_new_dset
  File "/media/david/ea1787b7-95f4-4875-ace4-8e79eb830ea3/home/david/Documents/Problema 3/Entornos/P3/lib/py

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGSEGV(-11)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
# ═══════════════════════════════════════════════════
# CELDA 8 — POST-PROCESAMIENTO: CALCULAR ETA
# ═══════════════════════════════════════════════════

# ── Consolidar archivos parciales de cada proceso ──
import glob
parciales = glob.glob(os.path.join(RESULTS_DIR, "parcial_*.csv"))
dfs = [pd.read_csv(p) for p in parciales]
df_consolidado = pd.concat(dfs, ignore_index=True)
df_consolidado.to_csv(CSV_PATH, index=False)
for p in parciales:
    os.remove(p)

# Leer CSV global consolidado
df = pd.read_csv(CSV_PATH)

def calcular_eficiencia(Jc_pos, Jc_neg):
    """
    Calcula la eficiencia del efecto diodo:
    η = |Jc+ - Jc-| / (Jc+ + Jc-) × 100
    """
    if pd.isna(Jc_pos) or pd.isna(Jc_neg):
        return np.nan
    if (Jc_pos + Jc_neg) == 0:
        return np.nan
    return abs(Jc_pos - Jc_neg) / (Jc_pos + Jc_neg) * 100


# Calcular eta para cada (N, B)
for N in N_PEINES:
    for B in B_VALORES:
        mask_pos = (df['N_peine'] == N) & (df['B_mT'] == B) & (df['polaridad'] == 'positivo')
        mask_neg = (df['N_peine'] == N) & (df['B_mT'] == B) & (df['polaridad'] == 'negativo')

        if mask_pos.any() and mask_neg.any():
            Jc1_pos = df.loc[mask_pos, 'Jc1_uA'].values[0]
            Jc1_neg = df.loc[mask_neg, 'Jc1_uA'].values[0]
            Jc2_pos = df.loc[mask_pos, 'Jc2_uA'].values[0]
            Jc2_neg = df.loc[mask_neg, 'Jc2_uA'].values[0]

            eta_c1 = calcular_eficiencia(Jc1_pos, Jc1_neg)
            eta_c2 = calcular_eficiencia(Jc2_pos, Jc2_neg)

            df.loc[mask_pos, 'eta_c1_pct'] = eta_c1
            df.loc[mask_pos, 'eta_c2_pct'] = eta_c2
            df.loc[mask_neg, 'eta_c1_pct'] = eta_c1
            df.loc[mask_neg, 'eta_c2_pct'] = eta_c2

# Guardar CSV actualizado con eta
df.to_csv(CSV_PATH, index=False)
print(f"CSV actualizado con eficiencias: {CSV_PATH}")
print(df.to_string(index=False))

: 

In [ ]:
# ═══════════════════════════════════════════════════
# CELDA 9 — RESUMEN FINAL CON TIEMPOS
# ═══════════════════════════════════════════════════

df = pd.read_csv(CSV_PATH)

exitosos = df[df['estado'] == 'OK']
fallidos  = df[df['estado'] != 'OK']

# Trabajo más rápido y más lento
if len(exitosos) > 0:
    idx_rapido = exitosos['tiempo_s'].idxmin()
    idx_lento  = exitosos['tiempo_s'].idxmax()
    rapido = exitosos.loc[idx_rapido]
    lento  = exitosos.loc[idx_lento]

horas   = int(t_global_total // 3600)
minutos = int((t_global_total % 3600) // 60)
segundos = int(t_global_total % 60)

print()
print("══════════════════════════════════════════════════")
print("          RESUMEN DE EJECUCIÓN FINAL")
print("══════════════════════════════════════════════════")
print(f"  Fin:              {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Tiempo total:     {t_global_total:.1f}s  →  {horas}h {minutos}m {segundos}s")
print(f"")
print(f"  Trabajos totales:  {len(TRABAJOS)}")
print(f"  Exitosos:          {len(exitosos)}/{len(TRABAJOS)}")
print(f"  Fallidos:          {len(fallidos)}/{len(TRABAJOS)}")
print(f"")
if len(exitosos) > 0:
    print(f"  Tiempo promedio por trabajo: {exitosos['tiempo_s'].mean():.1f}s")
    print(f"  Trabajo más rápido: N={rapido['N_peine']}, B={rapido['B_mT']}, {rapido['polaridad']} → {rapido['tiempo_s']}s")
    print(f"  Trabajo más lento:  N={lento['N_peine']},  B={lento['B_mT']}, {lento['polaridad']} → {lento['tiempo_s']}s")
print(f"")
print(f"  Resultados en:    {CSV_PATH}")
print(f"  Parámetros en:    {TXT_PATH}")
print("══════════════════════════════════════════════════")

if len(fallidos) > 0:
    print("\n  TRABAJOS FALLIDOS:")
    for _, row in fallidos.iterrows():
        print(f"  N={row['N_peine']}, B={row['B_mT']}, {row['polaridad']}: {row['estado']}")

# Añadir tiempo total al TXT de parámetros
with open(TXT_PATH, "a", encoding="utf-8") as f:
    f.write("\n--- RESULTADO GLOBAL ---\n")
    f.write(f"Fin:              {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Tiempo total:     {t_global_total:.1f}s ({horas}h {minutos}m {segundos}s)\n")
    f.write(f"Exitosos:         {len(exitosos)}/{len(TRABAJOS)}\n")
    f.write(f"Fallidos:         {len(fallidos)}/{len(TRABAJOS)}\n")

: 